# BusState Data Processing Pipeline (Python)

### Overview

This notebook processes raw **BusState APC data** from compressed `.txt.zip` files into clean, structured datasets for downstream analysis and dashboarding.

The workflow is a Python translation of an existing R-based pipeline previously used in the OSU Department of Transportation and Traffic Management (TTM), with improvements for modularity, readability, and scalability.

---

### Data Source

* Directory: `K:/AP/TTM/Data/APC Data/`
* File format: `compressed .txt`
* Naming convention:

  ```
  busstate0####DDMMYY.txt.zip
  ```

  * `####` = bus identifier
  * `DD` = day
  * `MM` = month
  * `YY` = year (2-digit)

---

### Output (`sort_and_save`)

* Data is split into monthly subsets

* Saved as `.csv` files:

  ```
  YYYY-MMM-busstate.csv
  ```

* Example:

  ```
  2025-OCT-busstate.csv
  ```

* Output directory:

  ```
  K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned/
  ```

---

### Notes

* This will take around 3-4 minutes to run for one month worth of data.
* If this is run on a Unix based machine, changes will need to be made to the root directory.
* Potential issues with overwriting in the sort_and_save function when running a new month with existing data. - will revisit

---

### Author

* Writen by Clayton Morgan (morgan.1461) Reporting and Analytics Analyst at The Ohio State University
* Python implementation of legacy R workflow in the TTM


In [1]:
import busstate_processing as bp
import dashboard_processing as dp
# note for future me: if you edit processing.py, run the lines below to reload w/out restarting kernel

# import importlib
# importlib.reload(bp)
# importlib.reload(dp)

**Note**: The year and month fields must be in 2 digit format, within the quotations.

e.g) For October, 2025

year = "25"

month = "09"

In [2]:
# STEP 1: Set month and year of interest for dashboard
year = "25"
month = "12"

In [3]:
# STEP 2: Run processing pipeline to clean busstate data for month and year of interest
bp.busstate_processing(year, month)

Starting busstate processing for 12/25...
Found 902 busstate files for 12/25 in 'K:/AP/TTM/Data/APC Data'
Unzipping and processing busstate files for 12/25...
Combined dataframe has 868244 records for 12/25
No data for month JAN 2025
No data for month FEB 2025
No data for month MAR 2025
No data for month APR 2025
No data for month MAY 2025
No data for month JUN 2025
No data for month JUL 2025
No data for month AUG 2025
Saved 9 records for SEP 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-SEP-busstate.csv'
No data for month OCT 2025
Saved 26107 records for NOV 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-NOV-busstate.csv'
Saved 842127 records for DEC 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-DEC-busstate.csv'
Finished processing busstate data for 12/25 in 180.94 seconds. Cleaned files saved to 'K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned'


### Stops inventory and metrics
This code will produce a dataframe containing the stops including the new University Hopsital and Doan Hall stops.

In [1]:
import pandas as pd
import numpy as np
import os
import pathlib

In [2]:
def build_stops_df():
    '''
    Build stops dataframe by merging static pattern stops and stop inventory files. Anytime that stops change, this will need to be re run after the csv files are updated.

    Parameters:
        None
    Returns:
        stops_df (pd.DataFrame): dataframe with stop id, stop name, and lat/lon coordinates for all stops in the pattern stops file
    '''
    # set directory paths for stop data - stored as 2 csv files in ./stops/
    stop_data_dir = pathlib.Path(os.getcwd()) / "stops"
    pattern_stops_path = stop_data_dir / "pattern_stops.csv"
    stop_inventory_path = stop_data_dir / "stop_inventory.csv"

    # read in static stop files
    pattern_stops = pd.read_csv(pattern_stops_path, header = None) # no header
    stop_inventory = pd.read_csv(stop_inventory_path)

    # only need cols 0 and 9 and can drop any dups
    pattern_stops = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
    pattern_stops.columns = ["ROUTE", "STOP_ID"] # rename cols for merge

    # Merge pattern stops and stop inventory on stop id - left merge 
    stops_df = pattern_stops.merge(stop_inventory, how = "left", on = "STOP_ID")

    return stops_df

In [3]:
def which_stop(lat, lon, route = "MC", max_distance = 0.0005):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: stops_df must be a global variable for this function to work, so build_stops_df() must be run before this function is called.

    Args:
        lat (float): The latitude of the point of interest.
        lon (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame. Potentially important in future for overlapping stops.
        max_distance (float): The maximum distance to consider for a stop. Default is 0.0005 per legacy R code. approximately 150ish feet

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # subset the stops to route of interest
    route_stops = stops_df[stops_df['ROUTE'] == route]

    distance_lon = route_stops['LONG'].values - lon
    distance_lat = route_stops['LAT'].values - lat

    # calculate the distance to each stop
    distances = np.sqrt(distance_lon**2 + distance_lat**2)

    mask = distances < max_distance
    # lowest distance should be the closest stop now. - each stop is at minimum approx 430ft apart so margin of 150 should be fine for MC route.
    selected_stop = route_stops[mask]

    # if no stops within alloted distance, return None
    if not np.any(mask):
        return None

    return int(route_stops.loc[mask, 'STOP_ID'].iloc[0])


In [4]:
# NOTE: Will need to update function with year and month functionality.
def process_mc_busstate():
    """
    Process the busstate data for the medical center route.
    NOTE: This will ONLY work for the medical center route.
    NOTE: This will return a significantly smaller dataframe
    Args:
        None
    Returns:
        DataFrame: A pandas DataFrame containing the processed busstate data for the medical center route.
    """
    year_full = "2025" # add function to convert, etc
    month_full = "DEC"

    busstate_dir = pathlib.Path(os.getcwd()) / "BusState Cleaned"
    busstate_path = os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv"))
    busstate_df = pd.read_csv(busstate_path) # Now cleaned busstate data read in

    # Process the busstate data for medical center routes
    #busstate_df.info()

    # should filter to just MC routes
    filtered_busstate_df = busstate_df.loc[(busstate_df['RUN_ID'] > 1500) & (busstate_df['RUN_ID'] < 1600)].copy()

    # Convert time metrics to datetime
    filtered_busstate_df['EVENT_TIME'] = pd.to_datetime(filtered_busstate_df['EVENT_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['DEPARTURE_TIME'] = pd.to_datetime(filtered_busstate_df['DEPARTURE_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['ENTER_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['ENTER_STOP_WINDOW_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['EXIT_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['EXIT_STOP_WINDOW_TIME'], format = "%H:%M:%S")

    # sort by date and event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['DATE', 'EVENT_TIME'])

    # Assign stop ID - most resource intensive step - as INT not float
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE']), axis = 1)

    # filter out 'dummy' stops, 27 and 461
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isin([27, 461])].index)

    # filter out empty stops - where bus was on route and not at a stop geographically
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isna()].index)

    # Convert to int
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df['STOP_ID'].astype(int)

    # filter by bus id, date, event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['BUS_ID', 'DATE', 'EVENT_TIME']).reset_index(drop = True) # restting index

    # create a new col for each new event
    # when stop changes OR bus_id changes OR elapsed time > 60
    count = (
        (filtered_busstate_df['STOP_ID'] != filtered_busstate_df['STOP_ID'].shift(1)) |
        (filtered_busstate_df['BUS_ID'] != filtered_busstate_df['BUS_ID'].shift(1)) |
        ((filtered_busstate_df['EVENT_TIME'] - filtered_busstate_df['EVENT_TIME'].shift(1)).dt.total_seconds() > 60)
    )

    # take cumsum of count to assign 
    filtered_busstate_df['COUNT'] = count.cumsum()

    # consolidate df
    consolidated_busstate_df = filtered_busstate_df.groupby(['BUS_ID', 'DATE', 'COUNT', 'STOP_ID', 'RUN_ID'], as_index = False).agg(
        BOARDINGS = ('BOARDINGS', 'max'),
        ALIGHTINGS = ('ALIGHTINGS', 'max'),
        LOAD = ('PASSENGER_LOAD', 'max'),
        EARLY_EVENT=("EVENT_TIME", "min"),
        LATE_EVENT=("EVENT_TIME", "max"),
        DEPARTURE_TIME=("DEPARTURE_TIME", "max"),
        ENTER_STOP=("ENTER_STOP_WINDOW_TIME", "min"),
        EXIT_STOP=("EXIT_STOP_WINDOW_TIME", "max"),
        RUN_ID = ("RUN_ID", "last"),
        DEST=("DEST_SIGN_ROUTE_TEXT", "last"),
    )
    #print(consolidated_busstate_df)

    # calculate the arrival and departure times for each event
    consolidated_busstate_df['ARRIVAL'] = consolidated_busstate_df[['EARLY_EVENT', 'ENTER_STOP']].min(axis=1) # arrival is earlier enter stop or early event
    consolidated_busstate_df['DEPARTURE'] = consolidated_busstate_df[['LATE_EVENT', 'DEPARTURE_TIME', 'EXIT_STOP']].max(axis=1) # departure is latest of late event or departure time or exit stop
    consolidated_busstate_df['DWELL'] = (consolidated_busstate_df['DEPARTURE'] - consolidated_busstate_df['ARRIVAL']) # total dwell time at stop

    # can drop unnecessary cols now
    consolidated_busstate_df = consolidated_busstate_df.drop(columns = ['EARLY_EVENT', 'LATE_EVENT', 'DEPARTURE_TIME', 'ENTER_STOP', 'EXIT_STOP'])

    # add hour and minute cols for filtering 
    consolidated_busstate_df['HOUR'] = consolidated_busstate_df['ARRIVAL'].dt.hour
    consolidated_busstate_df['MINUTE'] = consolidated_busstate_df['ARRIVAL'].dt.minute

    # ok - consolidated busstate df should now be ready for processing of the metrics.
    return consolidated_busstate_df

In [6]:
# This will produce df that metrics can be calculated from
stops_df = build_stops_df()

mc_busstate_consolidated = process_mc_busstate()

## Calculate the headways for each stop

In [81]:
mc_busstate_consolidated

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE
0,1301,2025-12-01,1,403,0,0,1,1512.0,MC,1900-01-01 05:50:04,1900-01-01 05:51:02,0 days 00:00:58,5,50
1,1301,2025-12-01,2,404,22,1,22,1512.0,MC,1900-01-01 05:52:01,1900-01-01 05:52:49,0 days 00:00:48,5,52
2,1301,2025-12-01,3,401,6,15,22,1512.0,MC,1900-01-01 05:57:46,1900-01-01 05:58:47,0 days 00:01:01,5,57
3,1301,2025-12-01,4,37,1,14,0,1512.0,MC,1900-01-01 05:59:21,1900-01-01 06:00:45,0 days 00:01:24,5,59
4,1301,2025-12-01,5,403,5,1,4,1512.0,MC,1900-01-01 06:07:08,1900-01-01 06:10:55,0 days 00:03:47,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36984,2503,2025-12-30,36964,404,1,9,1,1507.0,MC,1900-01-01 19:38:43,1900-01-01 19:40:51,0 days 00:02:08,19,38
36985,2503,2025-12-30,36965,94,0,2,0,1507.0,MC,1900-01-01 19:41:50,1900-01-01 19:42:17,0 days 00:00:27,19,41
36986,2503,2025-12-30,36966,95,0,0,0,1507.0,MC,1900-01-01 19:43:04,1900-01-01 19:43:31,0 days 00:00:27,19,43
36987,2503,2025-12-30,36967,401,0,0,0,1507.0,MC,1900-01-01 19:50:21,1900-01-01 19:50:33,0 days 00:00:12,19,50


In [82]:
# mc_busstate_consolidated['DWELL']
mc_busstate_consolidated['DATE'] = pd.to_datetime(mc_busstate_consolidated['DATE'], format = "%Y-%m-%d")
mc_busstate_consolidated.info()

<class 'pandas.DataFrame'>
RangeIndex: 36989 entries, 0 to 36988
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   BUS_ID      36989 non-null  int64          
 1   DATE        36989 non-null  datetime64[us] 
 2   COUNT       36989 non-null  int64          
 3   STOP_ID     36989 non-null  int64          
 4   BOARDINGS   36989 non-null  int64          
 5   ALIGHTINGS  36989 non-null  int64          
 6   LOAD        36989 non-null  int64          
 7   RUN_ID      36989 non-null  float64        
 8   DEST        36989 non-null  str            
 9   ARRIVAL     36989 non-null  datetime64[us] 
 10  DEPARTURE   36989 non-null  datetime64[us] 
 11  DWELL       36989 non-null  timedelta64[us]
 12  HOUR        36989 non-null  int32          
 13  MINUTE      36989 non-null  int32          
dtypes: datetime64[us](3), float64(1), int32(2), int64(6), str(1), timedelta64[us](1)
memory usage: 3.7 MB


In [9]:
def calculate_headway(busstate_df, stop_id):
    '''
    Calculate the headway for a given stop id in the busstate dataframe.

    Args:
        busstate_df (pd.DataFrame): The busstate dataframe containing the bus events.
        stop_id (int): The stop ID for which to calculate headways.
    Returns:
        pd.DataFrame: A subset dataframe containing the headways for the specified stop ID.
    '''
    # calculate headways of carmack 2 stop - 403
    stop_hw = busstate_df[busstate_df['STOP_ID'] == stop_id]

    stop_hw = stop_hw.sort_values(['DATE', 'ARRIVAL'])

    # stop_hw['ARRIVAL'].isna().sum() # zero NA
    stop_hw['HEADWAY'] = stop_hw['ARRIVAL'] - stop_hw['ARRIVAL'].shift(1) # headway in minutes

    stop_hw = stop_hw.loc[
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 >= 0) & # filter out negative headways
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 < 22) # filter out headways greater than 22 minutes - per legacy R code
        ] 

    # adjust for midnight arrivals where hour == 0
    stop_hw['DATE'] = stop_hw['DATE'].where(stop_hw['ARRIVAL'].dt.hour != 0,
                                                        stop_hw['DATE'] - pd.Timedelta(days = 1))
    stop_hw['HOUR'] = stop_hw['ARRIVAL'].dt.hour.where(stop_hw['ARRIVAL'].dt.hour != 0, 24)

    stop_hw.groupby(['DATE', 'ARRIVAL'])
    
    return stop_hw

In [83]:
carmack_2_id = 403
carmack_3_id = 404
university_hospital_id = 401
doan_hall_id = 37

carmack_2_hw = calculate_headway(mc_busstate_consolidated, carmack_2_id)
carmack_3_hw = calculate_headway(mc_busstate_consolidated, carmack_3_id)
university_hospital_hw = calculate_headway(mc_busstate_consolidated, university_hospital_id)
doan_hall_hw = calculate_headway(mc_busstate_consolidated, doan_hall_id)

# Now have headways for each stop in separate dataframes, can calculate metrics from here.
# If we wanted to calculate headway metrics BY STOP, we could do that here without concatenating

In [84]:
# combine all headways into one df for metrics calculation
combined_hw = pd.concat([carmack_2_hw, carmack_3_hw, university_hospital_hw, doan_hall_hw], ignore_index = True)

(combined_hw['HEADWAY'].dt.total_seconds() / 60).mean() # should be the average headway across all 4 stops for month of Dec in minutes.
(combined_hw['HEADWAY'].dt.total_seconds() / 60).median() # should be the median headway across all 4 stops for month of Dec in minutes.

np.float64(3.45)

In [12]:
combined_hw['HEADWAY'].describe()

count                     26226
mean     0 days 00:04:27.012049
std      0 days 00:04:00.855841
min             0 days 00:00:00
25%             0 days 00:01:44
50%             0 days 00:03:27
75%             0 days 00:06:00
max             0 days 00:21:59
Name: HEADWAY, dtype: object

In [ ]:
combined_hw.info()
combined_hw.isna().sum() # zero NA
combined_hw[combined_hw['HEADWAY'] == pd.Timedelta(seconds = 0)] # 73 total headways of 0 seconds
combined_hw[25400:25403] # 2 busses arrived at the same time at the same stop at doan - doesnt seem impossible?

<class 'pandas.DataFrame'>
RangeIndex: 26226 entries, 0 to 26225
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   BUS_ID      26226 non-null  int64          
 1   DATE        26226 non-null  datetime64[us] 
 2   COUNT       26226 non-null  int64          
 3   STOP_ID     26226 non-null  int64          
 4   BOARDINGS   26226 non-null  int64          
 5   ALIGHTINGS  26226 non-null  int64          
 6   LOAD        26226 non-null  int64          
 7   RUN_ID      26226 non-null  float64        
 8   DEST        26226 non-null  str            
 9   ARRIVAL     26226 non-null  datetime64[us] 
 10  DEPARTURE   26226 non-null  datetime64[us] 
 11  DWELL       26226 non-null  timedelta64[us]
 12  HOUR        26226 non-null  int32          
 13  MINUTE      26226 non-null  int32          
 14  HEADWAY     26226 non-null  timedelta64[us]
dtypes: datetime64[us](3), float64(1), int32(2), int64(6), str(1), ti

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,HEADWAY
25400,1904,2025-12-26,14215,37,0,0,3,1502.0,MC,1900-01-01 07:05:53,1900-01-01 07:07:09,0 days 00:01:16,7,5,0 days 00:04:21
25401,2501,2025-12-26,27782,37,0,3,0,1501.0,MC,1900-01-01 07:05:53,1900-01-01 07:06:45,0 days 00:00:52,7,5,0 days 00:00:00
25402,2401,2025-12-26,22144,37,0,3,0,1503.0,MC,1900-01-01 07:09:57,1900-01-01 07:10:54,0 days 00:00:57,7,9,0 days 00:04:04


In [86]:
def calculate_headway_dashboard_metrics(headway_df):
    '''
    Calculate headway metrics for each target hour, intended for external reporting use.
    NOTE: Target hour is stored as a constant within this function, so if target hours change, this function will need to be updated.

    Args:
        headway_df (pd.DataFrame): The combined headway dataframe containing headway information and target hours.
    Returns:
        pd.DataFrame: A summary dataframe containing the percentage of headways that met the target, as well as the 50th, 75th, and 90th percentile headway times for each target hour.
    '''
    # Target headways to calculate % on time for each timeframe
    TARGET_HEADWAYS = { # based on R code and report
        "5-6a": 10,
        "6-7a": 3,
        "7-8a": 3,
        "8a-2p": 10,
        "2-8p": 5,
        "8-10p": 10,
        "10p-12a": 5
    }
    # add in target headways to headway_df based on hour of arrival
    headway_df['TARGET'] = headway_df['HOUR'].apply(
        lambda hour: TARGET_HEADWAYS["5-6a"] if hour == 5 else (
            TARGET_HEADWAYS["6-7a"] if hour == 6 else (
                TARGET_HEADWAYS["7-8a"] if hour == 7 else (
                    TARGET_HEADWAYS["8a-2p"] if 8 <= hour < 14 else (
                        TARGET_HEADWAYS["2-8p"] if 14 <= hour < 20 else (
                            TARGET_HEADWAYS["8-10p"] if 20 <= hour < 22 else (
                                TARGET_HEADWAYS["10p-12a"] if 22 <= hour or hour == 0 else np.nan
                            )
                        )
                    )
                )
            )
        )
    )
    # create min of headway col to compare to target
    headway_df['HEADWAY_MIN'] = headway_df['HEADWAY'].dt.total_seconds() / 60

    # create met col for met headway target
    headway_df['MET'] = ((headway_df['TARGET'].notna()) & (headway_df['HEADWAY_MIN'] <= headway_df['TARGET'])).astype(int)
    # 1 if met, 0 if not met, only calculate if target is not NA

    # create target hour where each hour is broken into the target timeframes
    headway_df['TARGET_HOUR'] = headway_df['HOUR'].apply(
        lambda hour: "5-6a" if hour == 5 else (
            "6-7a" if hour == 6 else (
                "7-8a" if hour == 7 else (
                    "8a-2p" if 8 <= hour < 14 else (
                        "2-8p" if 14 <= hour < 20 else (
                            "8-10p" if 20 <= hour < 22 else (
                                "10p-12a" if 22 <= hour or hour == 0 else np.nan
                            )
                        )
                    )
                )
            )
        )
    )

    # combined_hw['HEADWAY_MIN'].head(20)
    # sort by chronological order
    target_order = list(TARGET_HEADWAYS.keys())
    headway_df['TARGET_HOUR'] = pd.Categorical(headway_df['TARGET_HOUR'], categories = target_order, ordered = True)

    # use HOUR col to create a metrics table with current combined data
    headway_summary = (
        headway_df
        .groupby('TARGET_HOUR') # grupby HOUR to have metrics by each individual hour, or TARGET_HOUR for the target timeframes
        .agg(
            MET = ("MET", "sum"),
            COUNT = ("HEADWAY", "size"),
            p50 = ("HEADWAY_MIN", lambda x: x.quantile(0.5)),
            p75 = ("HEADWAY_MIN", lambda x: x.quantile(0.75)),
            p90 = ("HEADWAY_MIN", lambda x: x.quantile(0.9)),
        )
    )
    # count here is LIKELY artificially inflated due to the 4th stop being added.
    headway_summary['MET'] = headway_summary['MET'] / headway_summary['COUNT'] # convert to percentage of headways that met target for each hour
    headway_summary.drop(columns = ['COUNT'], inplace = True)
    headway_summary.sort_values('TARGET_HOUR')

    return headway_summary
        

In [89]:
headway_summary = calculate_headway_dashboard_metrics(combined_hw)
headway_summary

,MET,p50,p75,p90
TARGET_HOUR,,,,
5-6a,0.959285,5.233333,6.850000,8.706667
6-7a,0.883029,1.183333,2.033333,3.250000
7-8a,0.721342,1.833333,3.250000,4.616667
8a-2p,0.908382,6.000000,8.116667,9.950000
2-8p,0.843833,2.933333,4.116667,5.583333
8-10p,0.963942,5.900000,7.337500,8.716667
10p-12a,0.511354,4.933333,6.483333,9.200000


In [94]:
def calculate_headway_internal_metrics(headway_df):
    '''
    Calculate internal headway metrics for each target hour, intended for internal use only and not external reporting. 
    This also can be configured to calculate metrics by stop if necessary, but currently is set up to calculate across all stops for each hour.
    This function is intended to be edited for future use as needed internally.

    Args:
        headway_df (pd.DataFrame): The combined headway dataframe containing headway information and target hours.
    Returns:
        pd.DataFrame: A summary dataframe containing headways by hour with summary statistcs includign mean, median, min, max, etc
    '''
    headway_summary = (
        headway_df
        .groupby('HOUR') # grupby HOUR to have metrics by each individual hour
        .agg(
            COUNT = ("HEADWAY", "size"),
            MEAN = ("HEADWAY_MIN", "mean"),
            MEDIAN = ("HEADWAY_MIN", "median"),
            MIN = ("HEADWAY_MIN", "min"),
            MAX = ("HEADWAY_MIN", "max"),
            p50 = ("HEADWAY_MIN", lambda x: x.quantile(0.5)),
            p75 = ("HEADWAY_MIN", lambda x: x.quantile(0.75)),
            p90 = ("HEADWAY_MIN", lambda x: x.quantile(0.9)),
        )
    )

    return headway_summary



In [ ]:
headway_internal_summary = calculate_headway_internal_metrics(combined_hw)
headway_internal_summary

,COUNT,MEAN,MEDIAN,MIN,MAX,p50,p75,p90
HOUR,,,,,,,,
1,317,16.815510,19.083333,0.050000,21.800000,19.083333,20.100000,20.746667
2,281,16.968446,19.633333,0.016667,21.983333,19.633333,20.733333,21.250000
3,333,16.262563,18.683333,0.000000,21.983333,18.683333,19.983333,21.083333
4,571,10.038850,9.883333,0.000000,21.700000,9.883333,11.466667,16.400000
5,1007,4.993512,5.233333,0.000000,18.316667,5.233333,6.850000,8.706667
6,3394,1.472505,1.183333,0.000000,21.450000,1.183333,2.033333,3.250000
7,2146,2.267016,1.833333,0.000000,21.983333,1.833333,3.250000,4.616667
8,1415,3.372980,2.583333,0.000000,13.816667,2.583333,4.966667,7.383333
9,776,5.974893,5.733333,0.016667,16.583333,5.733333,8.316667,10.483333


## Calculate the capacity metric

In [107]:
# combine UH and Doan stops for capacity
# Carmack -> UH is inbound, Doan -> Carmack is outbound
# Treating UH and Doan as one stop.
def combine_uh_doan_stops(df, max_gap_minutes = 5):
    """
    Combine University Hospital and Doan stops into a single stop in the stop inventory for metrics.

    Args:
        stop_inventory (DataFrame): A DataFrame containing the stop inventory information.
    Returns:
        DataFrame: A DataFrame with University Hospital and Doan stops combined into a single stop with ID of 999.
    """
    df = df.copy()

    # will keep the original stop column for UH and Doan for reference, but assign a new combined stop - potential for debugging purposes
    # Assigning the number 999 for combined stop - no actual meaning for this and an unused stop number
    df['STOP_ORIGINAL'] = df['STOP_ID'] # keep original stop id for reference

    doan_stop_id = 37
    uh_stop_id = 401
 
    combined_id = 999 # completely abritary 

    max_gap = pd.Timedelta(minutes=max_gap_minutes)

    sort_cols = ['BUS_ID', 'DATE', 'RUN_ID', 'ARRIVAL', 'DEPARTURE']
    working = df.sort_values(sort_cols).reset_index(drop=True)

    combined_rows = []
    row_index = 0

    while row_index < len(working):
        current = working.iloc[row_index]

    if row_index < len(working) - 1:
        next_row = working.iloc[row_index + 1]

        is_candidate_pair = (
            current['STOP_ID'] == uh_stop_id
            and next_row['STOP_ID'] == doan_stop_id
            and current['BUS_ID'] == next_row['BUS_ID']
            and current['DATE'] == next_row['DATE']
            and current['RUN_ID'] == next_row['RUN_ID']
            and (next_row['ARRIVAL'] - current['DEPARTURE']) <= max_gap
            and (next_row['ARRIVAL'] - current['DEPARTURE']) >= pd.Timedelta(0)
        )

        if is_candidate_pair:
            merged = current.copy()
            merged['STOP_ID'] = combined_id
            merged['STOP_ORIGINAL'] = f"{uh_stop_id}_{doan_stop_id}"
            merged['BOARDINGS'] = current['BOARDINGS'] + next_row['BOARDINGS']
            merged['ALIGHTINGS'] = current['ALIGHTINGS'] + next_row['ALIGHTINGS']
            merged['LOAD'] = max(current['LOAD'], next_row['LOAD'])
            merged['DEPARTURE'] = next_row['DEPARTURE']
            merged['DWELL'] = merged['DEPARTURE'] - merged['ARRIVAL']
            merged['HOUR'] = merged['ARRIVAL'].hour
            merged['MINUTE'] = merged['ARRIVAL'].minute
            combined_rows.append(merged)
            row_index += 2

    combined_rows.append(current.copy())
    row_index += 1

    return pd.DataFrame(combined_rows).reset_index(drop=True)


#mc_rider = combine_uh_doan_stops(mc_busstate_consolidated)

In [ ]:
subset_mc_busstate_consolidated = mc_busstate_consolidated.head(50)

subset_mc_busstate_consolidated

mc_rider = combine_uh_doan_stops(subset_mc_busstate_consolidated)

In [ ]:
# Filtering for combined stop of UH and Doan for ridership
mc_rider_combined_stop = mc_rider[mc_rider['STOP_ID'] == 999]
# overwrite existing load col with the max load 
# need to find a way to combine the boarding and alightings for each individual bus across stop 401 and 37...


,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,STOP_ORIGINAL
2,1301,2025-12-01,3,999,6,15,15,1512.0,MC,1900-01-01 05:57:46,1900-01-01 05:58:47,0 days 00:01:01,5,57,401
3,1301,2025-12-01,4,999,1,14,14,1512.0,MC,1900-01-01 05:59:21,1900-01-01 06:00:45,0 days 00:01:24,5,59,37
6,1301,2025-12-01,7,999,0,2,2,1512.0,MC,1900-01-01 06:18:31,1900-01-01 06:18:59,0 days 00:00:28,6,18,401
7,1301,2025-12-01,8,999,2,13,13,1512.0,MC,1900-01-01 06:19:25,1900-01-01 06:21:51,0 days 00:02:26,6,19,37
10,1301,2025-12-01,11,999,0,12,12,1512.0,MC,1900-01-01 06:38:43,1900-01-01 06:39:28,0 days 00:00:45,6,38,401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36976,2503,2025-12-30,36956,999,7,2,7,1507.0,MC,1900-01-01 19:07:42,1900-01-01 19:09:18,0 days 00:01:36,19,7,37
36981,2503,2025-12-30,36961,999,3,0,3,1507.0,MC,1900-01-01 19:27:56,1900-01-01 19:29:26,0 days 00:01:30,19,27,401
36982,2503,2025-12-30,36962,999,11,1,11,1507.0,MC,1900-01-01 19:29:44,1900-01-01 19:31:30,0 days 00:01:46,19,29,37
36987,2503,2025-12-30,36967,999,0,0,0,1507.0,MC,1900-01-01 19:50:21,1900-01-01 19:50:33,0 days 00:00:12,19,50,401
